# Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import os

In [ ]:
# ===================================================
# Google Drive Mount & Unzip to Colab Local Storage
# ===================================================
import zipfile
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define paths (Extraction will be done in Colab local storage)
ZIP_PATH = "/content/drive/MyDrive/Dataset.zip"
EXTRACT_PATH = "/content/"  # <--- This is now the current directory of Colab

print("Unzipping from Google Drive to Colab has started...")

try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Dataset has been successfully extracted to the current Colab directory!")

except zipfile.BadZipFile:
    print("Error: The zip file is corrupt or was not uploaded correctly to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive se Colab mein unzipping shuru ho rahi hai...
Dataset kamyabi se Colab ki current directory mein extract ho gaya hai!



# Dataset Path


In [ ]:
DATASET_PATH = r"Dataset"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


# Train/Test Split (80/20)


In [7]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

print("\nClass Labels:")
for i, cls in enumerate(class_names):
    print(f"{i} -> {cls}")

NUM_CLASSES = len(class_names)

Found 7000 files belonging to 7 classes.
Using 5600 files for training.
Found 7000 files belonging to 7 classes.
Using 1400 files for validation.

Class Labels:
0 -> Bacterial Blight
1 -> Curl Virus
2 -> Healthy Leaf
3 -> Herbicide Growth Damage
4 -> Leaf Hopper Jassids
5 -> Leaf Redding
6 -> Leaf Variegation


# Save Class Labels with Numbers

In [8]:

labels_path = "labels.txt"
with open(labels_path, "w") as f:
    for i, cls in enumerate(class_names):
        f.write(f"{i} -> {cls}\n")

print(f"Labels with numbers saved in '{labels_path}'.")

Labels numbers ke sath 'labels.txt' mein save ho gaye hain!



# Performance Optimization


In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


# Data Augmentation Layer


In [10]:

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])



# MobileNetV2 Base Model


In [11]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step



# Transfer Learning Model


In [12]:

inputs = tf.keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation='softmax'
)(x)

model = tf.keras.Model(inputs, outputs)


# Compile


In [13]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,266,951 (8.65 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


# Callbacks


In [14]:
callbacks = [

    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2
    ),

    ModelCheckpoint(
        "best_model.keras",
        save_best_only=True
    )
]


# Initial Training


In [15]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
175/175 ━━━━━━━━━━━━━━━━━━━━ 56s 131ms/step - accuracy: 0.6482 - loss: 1.0070 - val_accuracy: 0.8143 - val_loss: 0.5717 - learning_rate: 0.0010
Epoch 2/15
175/175 ━━━━━━━━━━━━━━━━━━━━ 43s 69ms/step - accuracy: 0.8121 - loss: 0.5387 - val_accuracy: 0.8293 - val_loss: 0.5140 - learning_rate: 0.0010
Epoch 3/15
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 70ms/step - accuracy: 0.8439 - loss: 0.4562 - val_accuracy: 0.8600 - val_loss: 0.4054 - learning_rate: 0.0010
Epoch 4/15
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 70ms/step - accuracy: 0.8645 - loss: 0.3977 - val_accuracy: 0.8593 - val_loss: 0.3947 - learning_rate: 0.0010
Epoch 5/15
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 71ms/step - accuracy: 0.8768 - loss: 0.3686 - val_accuracy: 0.8821 - val_loss: 0.3267 - learning_rate: 0.0010
Epoch 6/15
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 70ms/step - accuracy: 0.8825 - loss: 0.3512 - val_accuracy: 0.8843 - val_loss: 0.3235 - learning_rate: 0.0010
Epoch 7/15
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 69ms/step - accuracy: 0.8863 - 


# Fine Tuning


In [16]:

base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 27s 98ms/step - accuracy: 0.7914 - loss: 0.6345 - val_accuracy: 0.8857 - val_loss: 0.3168 - learning_rate: 1.0000e-05
Epoch 2/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 87ms/step - accuracy: 0.8537 - loss: 0.4158 - val_accuracy: 0.8957 - val_loss: 0.2985 - learning_rate: 1.0000e-05
Epoch 3/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accuracy: 0.8723 - loss: 0.3730 - val_accuracy: 0.8943 - val_loss: 0.2864 - learning_rate: 2.0000e-06
Epoch 4/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8788 - loss: 0.3451 - val_accuracy: 0.9014 - val_loss: 0.2748 - learning_rate: 2.0000e-06
Epoch 5/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accuracy: 0.8838 - loss: 0.3325 - val_accuracy: 0.9021 - val_loss: 0.2704 - learning_rate: 4.0000e-07


In [17]:

# =====================================
# Final Evaluation
# =====================================

loss, acc = model.evaluate(test_ds)

print(f"\nFinal Accuracy: {acc*100:.2f}%")

# =====================================
# Save Model
# =====================================

model.save("cotton_leaf_mobilenetv2.keras")

44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.8857 - loss: 0.3168

Final Accuracy: 88.57%



# TFLite Conversion + Optimization



In [19]:
import tensorflow as tf
import os

# Load Saved Model
model = tf.keras.models.load_model(
    "cotton_leaf_mobilenetv2.keras"
)


# Float16 Quantization


In [20]:

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.target_spec.supported_types = [
    tf.float16
]

tflite_quant_model = converter.convert()

with open(
    "cotton_leaf_mobilenetv2_float16.tflite",
    "wb"
) as f:
    f.write(tflite_quant_model)

print("Float16 Quantized Model Saved")

Saved artifact at '/tmp/tmp5hs0or3c'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
Captures:
  135274583409296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583412944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583412752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583412368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583413520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583412176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583413136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583413328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583409104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135274583414480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1352745834139


# Model Size Comparison


In [21]:


tflite_path="cotton_leaf_mobilenetv2_float16.tflite"

keras_size = os.path.getsize(
    "cotton_leaf_mobilenetv2.keras"
) / (1024 * 1024)

tflite_size = os.path.getsize(
    tflite_path
) / (1024 * 1024)

print("\nModel Saved Successfully")
print(f"Keras Model Size  : {keras_size:.2f} MB")
print(f"TFLite Model Size : {tflite_size:.2f} MB")


Model Saved Successfully
Keras Model Size  : 20.96 MB
TFLite Model Size : 4.27 MB
